# Comparator Table

This notebook reports reference models that help interpret the primary SRM composites.

**Comparator roles**

| Comparator | Nature | Input | Output | Why included |
|---|---|---|---|---|
| LDA | Linear Discriminant Analysis visit-separation direction | Imaging visit rows | Projection score | Tests whether visits can be separated by imaging patterns. |
| Regression reference | ElasticNet clinical target model | Imaging visit rows | Predicted clinical score | Tests whether imaging predicts clinical scales used as benchmarks. |

**Interpretation warning:** LDA Fisher-criterion separation and paired SRM-based progression `d_z` are related but not directly interchangeable.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import DEFAULT_CONFIG, set_global_seeds
from src.data.audit import modelling_pair_count_table
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.cv import interaction_loocv, lda_loocv, lda_nested_loocv, tune_and_run_regression_loocv
from src.eval.intervals import adjacent_pair_interval_effect_summary, annual_tuning_diagnostics
from src.eval.metrics import bootstrap_ci_d, clinical_change_effect_sizes, reference_effect_sizes
from src.eval.optimization import optimization_log, optimization_row, save_optimization_log
from src.reporting.tuning_review import tuning_recommendation, tuning_verification_summary
from src.reporting.fold_comparison import fold_train_test_clinical_benchmark_table
from src.features.selection import feature_stability_report
from src.models.srm_global import srm_global_loocv

set_global_seeds(DEFAULT_CONFIG.random_state)
pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
if not pairs_path.exists():
    raise FileNotFoundError(f"Required modelling dataset not found: {pairs_path}")
pairs_df = pd.read_csv(pairs_path)

modelling_counts = modelling_pair_count_table(pairs_df, expected={"N12": 108, "N23": 99, "N13": 90, "N123": 90})
print("Canonical modelling-cohort pair counts")
display(modelling_counts)
long_df = trackfa_pairs_to_long(pairs_df)
groups = infer_trackfa_feature_groups(pairs_df)
imaging_cols = [c for c in groups.all_neuroimaging if c in long_df.columns]
subject_col = "pair_id"  # progression interval id, e.g. AAN001_V1V2
split_group_col = "subject"  # participant id; keeps V1V2 and V2V3 in the same fold
selection_method = "none"
selection_k = 8
LDA_CV_N_SPLITS = DEFAULT_CONFIG.cv_n_splits  # Subject-level grouped CV for annual-consistency tuning.
REGRESSION_CV_N_SPLITS = DEFAULT_CONFIG.cv_n_splits
N_BOOT = 300
RANDOM_SEED = DEFAULT_CONFIG.random_state
RIDGE_GRID = [1e-8, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]
COVARIANCE_SHRINKAGE_GRID = [0.75, 1.0]
LDA_Z_CLIP_GRID = [None, 4.0]
ELASTICNET_L1_RATIO_GRID = [0.2, 0.5, 0.8, 1.0]
REGRESSION_Z_CLIP_GRID = [None, 4.0]
LDA_SHRINK_GRID = ["auto", 1e-8, 0.1, 1.0, 10.0]
REGRESSION_PARAM_SELECTION_METRIC = "annual_mean_dz"
REGRESSION_MODEL_KINDS = ["elasticnet"]
optimization_rows = []
print(f"Loaded {pairs_path.name}: {long_df.shape[0]} visit rows, {len(imaging_cols)} imaging features")
print({"selection_method": selection_method, "selection_k": selection_k, "lda_cv_n_splits": LDA_CV_N_SPLITS,
    "regression_cv_n_splits": REGRESSION_CV_N_SPLITS, "regression_param_selection_metric": REGRESSION_PARAM_SELECTION_METRIC})
def benchmark_table(model_name: str, d_score: float, ci_low: float, ci_high: float) -> pd.DataFrame:
    imaging_ref = reference_effect_sizes(
        long_df,
        imaging_cols=imaging_cols,
        scale_cols=(),
        subject_col=subject_col,
        visit_col="visit",
    )
    clinical_ref = clinical_change_effect_sizes(
        pairs_df,
        scale_cols=("FARS", "SARA"),
        pair_types=("V1V2", "V2V3"),
    )
    rows = [{"feature": model_name, "kind": "model", "d": d_score, "ci_low": ci_low, "ci_high": ci_high, "source_delta_col": np.nan, "pair_types": np.nan}]
    for scale in ("FARS", "SARA"):
        hit = clinical_ref[(clinical_ref["kind"] == "scale") & (clinical_ref["feature"] == scale)].head(1)
        if len(hit):
            r = hit.iloc[0].to_dict()
            rows.append({"feature": r["feature"], "kind": r["kind"], "d": r["d"], "ci_low": np.nan, "ci_high": np.nan, "source_delta_col": r.get("source_delta_col", np.nan), "pair_types": r.get("pair_types", np.nan)})
    top_img = imaging_ref[imaging_ref["kind"] == "imaging"].head(1)
    if len(top_img):
        r = top_img.iloc[0].to_dict()
        rows.append({"feature": r["feature"], "kind": r["kind"], "d": r["d"], "ci_low": np.nan, "ci_high": np.nan, "source_delta_col": r.get("source_delta_col", np.nan), "pair_types": r.get("pair_types", np.nan)})
    return pd.DataFrame(rows)


In [ ]:
# Fold-level clinical train/test benchmark for supervisor review.
fold_train_test_clinical_benchmarks = fold_train_test_clinical_benchmark_table(
    long_df,
    pairs_df,
    imaging_cols,
    subject_col=subject_col,
    visit_col="visit",
    split_group_col=split_group_col,
    cv_n_splits=REGRESSION_CV_N_SPLITS,
    random_seed=RANDOM_SEED,
    clinical_scales=("FARS", "SARA"),
    pair_types=("V1V2", "V2V3"),
)
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
fold_train_test_clinical_benchmarks.to_csv(
    RESULTS_DIR / "comparator_fold_train_test_clinical_benchmarks.csv",
    index=False,
)
clinical_display_cols = [
    "fold",
    "clinical_scale",
    "train_n_subjects",
    "test_n_subjects",
    "clinical_train_n_pairs",
    "clinical_test_n_pairs",
    "clinical_train_d",
    "clinical_test_d",
    "clinical_train_minus_test_d",
]
print("Comparator fold-level train/test clinical benchmarks")
display(fold_train_test_clinical_benchmarks[clinical_display_cols])


## 1. LDA Visit-Separation Comparator

Linear Discriminant Analysis (LDA) finds a direction that separates visit labels. It is useful as a comparator, but the project target remains paired progression change on held-out participant groups.


In [ ]:
import time

lda_trials = []
for z_clip in LDA_Z_CLIP_GRID:
    for covariance_shrinkage in COVARIANCE_SHRINKAGE_GRID:
        for shrink in LDA_SHRINK_GRID:
            start = time.time()
            res = lda_loocv(
                long_df,
                imaging_cols,
                subject_col=subject_col,
                visit_col="visit",
                selection_method=selection_method,
                k=selection_k,
                cv_n_splits=LDA_CV_N_SPLITS,
                random_seed=RANDOM_SEED,
                split_group_col=split_group_col,
                shrink=shrink,
                covariance_shrinkage=covariance_shrinkage,
                z_clip=z_clip,
                compute_ci=False,
            )
            interval_summary = adjacent_pair_interval_effect_summary(
                res["oof_df"],
                pair_col=subject_col,
                visit_col="visit",
                score_col="score",
                n_boot=N_BOOT,
                seed=RANDOM_SEED,
            )
            res = {**res, **annual_tuning_diagnostics(interval_summary)}
            row = optimization_row(
                model="LDA exploratory",
                params={
                    "shrink": shrink,
                    "covariance_shrinkage": covariance_shrinkage,
                    "z_clip": z_clip,
                    "selection_method": selection_method,
                    "regularization": "ridge_plus_covariance_shrinkage_plus_optional_z_clip",
                },
                result=res,
                runtime_sec=time.time() - start,
                notes="exploratory grid reported with annual mean d_z and V1->V2/V2->V3 consistency diagnostics",
            )
            lda_trials.append((res, row))
            optimization_rows.append(row)

lda_optimization_df = optimization_log([row for _, row in lda_trials], sort_by="mean_validation_annual_dz")
print("LDA tuning candidates evaluated:", len(lda_optimization_df))

lda_review = tuning_recommendation(lda_optimization_df)
print("Numerically best LDA configuration")
display(pd.DataFrame([lda_review["raw_best"]]))
print("One-SE / near-optimal LDA candidate count:", len(lda_review["near_optimal"]))
print("Recommended LDA configuration by implemented hierarchy")
display(pd.DataFrame([lda_review["recommended"]]))
print(lda_review["summary"])
print("Human-verification summary")
display(tuning_verification_summary(lda_review))

lda_nested_candidates = [
    {
        "shrink": shrink,
        "covariance_shrinkage": covariance_shrinkage,
        "z_clip": z_clip,
        "selection_method": selection_method,
        "k": selection_k,
    }
    for z_clip in LDA_Z_CLIP_GRID + [3.0]
    for covariance_shrinkage in COVARIANCE_SHRINKAGE_GRID
    for shrink in LDA_SHRINK_GRID
]
start = time.time()
lda_res = lda_nested_loocv(
    long_df,
    imaging_cols,
    subject_col=subject_col,
    visit_col="visit",
    candidates=lda_nested_candidates,
    cv_n_splits=LDA_CV_N_SPLITS,
    inner_folds=5,
    random_seed=RANDOM_SEED,
    split_group_col=split_group_col,
    compute_ci=True,
    tuning_metric="annual_mean_dz",
)
lda_interval_summary = adjacent_pair_interval_effect_summary(
    lda_res["oof_df"],
    pair_col=subject_col,
    visit_col="visit",
    score_col="score",
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)
lda_res = {**lda_res, **annual_tuning_diagnostics(lda_interval_summary)}
optimization_rows.append(optimization_row(
    model="LDA nested",
    params={"candidate_count": len(lda_nested_candidates), "inner_folds": 5, "tuning": "train-fold inner grouped CV"},
    result=lda_res,
    runtime_sec=time.time() - start,
    notes="nested estimate with annual interval diagnostics; tune/report mean annual d_z and interval gap",
))
display(lda_res["chosen_params_df"].head())
display(lda_interval_summary)
pd.DataFrame([{
    "model": "LDA nested",
    "selection_method": selection_method,
    "candidate_count": len(lda_nested_candidates),
    "dz_v1_v2": lda_res["dz_v1_v2"],
    "dz_v2_v3": lda_res["dz_v2_v3"],
    "mean_annual_d_z": lda_res["mean_validation_annual_dz"],
    "annual_interval_gap": lda_res["annual_interval_gap"],
    "pooled_pair_d_z_reference": lda_res["d_score"],
    "ci_low": lda_res["d_ci_low"],
    "ci_high": lda_res["d_ci_high"],
    "n_subject_pairs": lda_res["n_subjects"],
}])


## 2. Regression Reference

Regression models predict clinical targets from imaging features. These outputs are references, not the primary biomarker objective, because the project aims to measure imaging progression rather than optimise clinical-score prediction.


In [ ]:
import time

regression_rows = []
regression_results = {}
for target in [c for c in ["FARS", "SARA"] if c in long_df.columns]:
    for model_kind in REGRESSION_MODEL_KINDS:
        for z_clip in REGRESSION_Z_CLIP_GRID:
            start = time.time()
            res = tune_and_run_regression_loocv(
                long_df,
                imaging_cols,
                target_col=target,
                subject_col=subject_col,
                model_kind=model_kind,
                selection_method=selection_method,
                k=selection_k,
                visit_col="visit",
                cv_n_splits=REGRESSION_CV_N_SPLITS,
                param_selection_metric=REGRESSION_PARAM_SELECTION_METRIC,
                z_clip=z_clip,
                split_group_col=split_group_col,
            )
            interval_summary = adjacent_pair_interval_effect_summary(
                res["oof_df"],
                pair_col=subject_col,
                visit_col="visit",
                score_col="pred",
                n_boot=N_BOOT,
                seed=RANDOM_SEED,
            )
            annual_diag = annual_tuning_diagnostics(interval_summary)
            res = {**res, **annual_diag}
            regression_results[(target, model_kind, z_clip)] = res
            params = {
                "target": target,
                "model_kind": model_kind,
                "selection_method": selection_method,
                "param_selection_metric": REGRESSION_PARAM_SELECTION_METRIC,
                "z_clip": z_clip,
            }
            optimization_rows.append(optimization_row(
                model=f"Regression {model_kind}",
                params=params,
                result=res,
                runtime_sec=time.time() - start,
                notes="inner hyperparameters reported with annual mean d_z and V1->V2/V2->V3 consistency diagnostics",
            ))
            regression_rows.append({
                "target": target,
                "model": model_kind,
                "selection_method": selection_method,
                "param_selection_metric": REGRESSION_PARAM_SELECTION_METRIC,
                "z_clip": z_clip,
                "rmse": res["rmse"],
                "r2": res["r2"],
                "dz_v1_v2": res["dz_v1_v2"],
                "dz_v2_v3": res["dz_v2_v3"],
                "mean_annual_d_z": res["mean_validation_annual_dz"],
                "annual_interval_gap": res["annual_interval_gap"],
                "p_progression": res["p_progression"],
                "pooled_pair_d_z_reference": res["d_score"],
                "ci_low": res["d_ci_low"],
                "ci_high": res["d_ci_high"],
                "n_subject_pairs": res["n_subjects"],
            })
regression_df = pd.DataFrame(regression_rows)
optimization_df = optimization_log(optimization_rows, sort_by="mean_validation_annual_dz")
log_path = save_optimization_log(optimization_df, REPO_ROOT / "results" / "comparator_optimization_log.csv")
print("Saved optimization log:", log_path)
display(regression_df.sort_values(["mean_annual_d_z", "annual_interval_gap"], ascending=[False, True]).groupby(["target", "model"], as_index=False, sort=False).head(1))
print("Comparator tuning candidates evaluated:", len(optimization_df))

for review_model, review_group in optimization_df.groupby("model", sort=False):
    model_review = tuning_recommendation(review_group)
    print(f"Tuning review: {review_model}")
    print("Numerically best configuration")
    display(pd.DataFrame([model_review["raw_best"]]))
    print("One-SE / near-optimal candidate set")
    display(model_review["near_optimal"])
    print("Recommended configuration by implemented hierarchy")
    display(pd.DataFrame([model_review["recommended"]]))
    print(model_review["summary"])
    print("Human-verification summary")
    display(tuning_verification_summary(model_review))


## 3. Clinical Benchmark Table

The final display compares the best comparator row with FARS, SARA, and the top single imaging feature using the same paired Cohen's `d_z` convention where applicable.


In [ ]:
best_reg = regression_df.sort_values(["mean_annual_d_z", "annual_interval_gap"], ascending=[False, True]).head(1)
if len(best_reg):
    row = best_reg.iloc[0]
    model_name = f"Regression {row['model']} ({row['target']})"
    model_d = row["pooled_pair_d_z_reference"]
    model_lo = row["ci_low"]
    model_hi = row["ci_high"]
else:
    model_name, model_d, model_lo, model_hi = "Regression reference", np.nan, np.nan, np.nan
lda_table = pd.DataFrame([{"feature": "LDA nested", "kind": "model", "d": lda_res["d_score"], "ci_low": lda_res["d_ci_low"], "ci_high": lda_res["d_ci_high"]}])
reg_table = benchmark_table(model_name, model_d, model_lo, model_hi)
display(pd.concat([lda_table, reg_table], ignore_index=True))
